In [ ]:
import ee
import geemap
import random
from s2sphere import LatLng, LatLngRect, RegionCoverer, Cell, CellId
from shapely.geometry import shape, Polygon, MultiPolygon, Point
import random
import csv

In [2]:
ee.Authenticate()
ee.Initialize()

In [3]:
aez = ee.FeatureCollection(
    "projects/ext-datasets/assets/datasets/Agro_Ecological_Zones"
)
india_ee_geom = aez.geometry().dissolve()

india_geojson = india_ee_geom.getInfo()
india_shape = shape(india_geojson)

In [ ]:
polygons = []
if isinstance(india_shape, Polygon):
    polygons = [india_shape]
elif isinstance(india_shape, MultiPolygon):
    polygons = list(india_shape.geoms)
else:
    for g in india_shape.geoms:
        if isinstance(g, (Polygon, MultiPolygon)):
            polygons.extend(list(g.geoms) if isinstance(g, MultiPolygon) else [g])

print("Total India polygons:", len(polygons))


def polygon_to_s2_cells(poly, level=13):
    minx, miny, maxx, maxy = poly.bounds
    rect = LatLngRect.from_point_pair(
        LatLng.from_degrees(miny, minx),
        LatLng.from_degrees(maxy, maxx)
    )
    coverer = RegionCoverer()
    coverer.min_level = level
    coverer.max_level = level
    coverer.max_cells = 100000
    return coverer.get_covering(rect)


s2_ids = set()
for poly in polygons:
    cells = polygon_to_s2_cells(poly, level=13)
    for c in cells:
        cell = Cell(c)  
        latlng = LatLng.from_point(cell.get_center()) 
        point = Point(latlng.lng().degrees, latlng.lat().degrees) 
        if poly.contains(point): 
            s2_ids.add(c.id())

print("Total S2 cells inside India:", len(s2_ids))


def s2cell_to_ee_polygon(cell_id_int):
    cell = Cell(CellId(cell_id_int))
    coords = []
    for i in range(4):
        v = cell.get_vertex(i)
        ll = LatLng.from_point(v)
        coords.append([ll.lng().degrees, ll.lat().degrees])
    coords.append(coords[0])
    return ee.Feature(ee.Geometry.Polygon([coords]))


Total India polygons: 587
Total S2 cells inside India: 2256839


In [13]:
s2_ids_list = list(s2_ids)
sample_ids = random.sample(s2_ids_list, 3000)

ee_features = [s2cell_to_ee_polygon(cid) for cid in sample_ids]
s2_fc = ee.FeatureCollection(ee_features)

Map = geemap.Map(center=[22, 80], zoom=4)
Map.addLayer(india_ee_geom, {"color": "black"}, "India boundary")
Map.addLayer(s2_fc, {"color": "red"}, "S2 cell centers (sample 1000)")
Map

Map(center=[22, 80], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

In [9]:
rows = []

for cid in s2_ids:
    cell = Cell(CellId(cid))
    coords = []

    for i in range(4):
        ll = LatLng.from_point(cell.get_vertex(i))
        coords.append(f"{ll.lng().degrees} {ll.lat().degrees}")

    coords.append(coords[0])  # close polygon

    wkt = "POLYGON ((" + ", ".join(coords) + "))"

    rows.append({
        "s2_id": f"id_{cid}",
        "wkt": wkt
    })

with open("s2_level13_india.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["s2_id", "wkt"])
    writer.writeheader()
    writer.writerows(rows)